# Generación y reutilización del dataset contextual

Por defecto carga los prototipos verificados e inspecciona los metadatos. No descarga corpus ni ejecuta BERT. Para las ocurrencias, copia primero el Parquet indicado en el README.


In [1]:
from pathlib import Path
import sys

def localizar_base():
    for parent in (Path.cwd(), *Path.cwd().parents):
        for candidate in (parent, parent / 'EstudioContextual/GeneracionDataset', parent / 'GeneracionDataset'):
            if (candidate / 'Codigo/contextual.py').is_file():
                return candidate.resolve()
    raise FileNotFoundError('Abre el notebook desde ProyectoTesis o una de sus subcarpetas.')

BASE = localizar_base()
sys.path.insert(0, str(BASE / 'Codigo'))
import contextual as ctx
print('Carpeta:', BASE)


Carpeta: C:\Users\jisaa\Documents\ProyectoTesis\EstudioContextual\GeneracionDataset


In [2]:
PROTOTIPOS = ctx.cargar_prototipos(BASE)
print(ctx.verificar_prototipos(PROTOTIPOS))
display(PROTOTIPOS.head())


{'shape': [50455, 768], 'vocabulary_sha256': '95d87d7ce442dd36699ab0ff7bdd8bd12c10e5460f642f6ba4a514369e088edc', 'embedding_sha256': '0590ffe496e1c0da00b399e4773d715f835b17ce2c0b0004ff19fcd3c7269593', 'status': 'VERIFIED'}


,dim_000,dim_001,dim_002,dim_003,dim_004,dim_005,dim_006,dim_007,dim_008,dim_009,...,dim_758,dim_759,dim_760,dim_761,dim_762,dim_763,dim_764,dim_765,dim_766,dim_767
word,,,,,,,,,,,,,,,,,,,,,
aaa,-0.626143,1.275576,0.229709,-0.319032,-0.205193,-0.700326,0.617611,-0.388610,-0.070717,-0.438661,...,-0.568987,0.243059,-0.068187,0.181786,0.294033,-1.098189,-0.233224,-0.382484,0.292886,-0.205680
aaas,-0.391977,0.969970,-0.102125,-0.293156,-0.055839,-1.191735,0.146250,-0.454013,-0.564199,-0.016485,...,-0.554876,0.341529,-0.552842,-0.503014,0.118255,-0.555671,-0.231098,-0.250074,0.161809,0.138633
aachen,0.722870,0.519042,0.324281,-0.372950,0.290919,-0.226253,0.252298,0.498785,0.122156,-0.013926,...,0.186095,0.575092,0.123500,-0.173290,0.206542,-0.625113,-0.311080,-0.302016,-0.135054,0.163212
aah,0.374000,1.219937,0.418105,-0.656825,0.740470,-0.629976,0.639615,-0.027693,0.458845,0.180718,...,-0.886168,-0.379242,0.038729,-0.691849,-0.497102,-0.633697,-0.055966,-0.424343,0.133243,-0.438860
aalborg,-0.147610,0.323266,0.816178,-0.500001,0.461349,-0.785293,0.115092,0.203538,-0.808534,-0.144945,...,-0.019378,0.348506,0.270836,-0.204014,0.238492,-0.577368,-0.257552,-0.250571,-0.067769,0.331198


In [3]:
METADATOS = ctx.verificar_metadatos(BASE)
print(METADATOS)
PARQUET = BASE / 'Datos/Ocurrencias/contextual_occurrences_definitive.parquet'


{'master_words': 103029, 'expected_rows': 10302900, 'reports_and_index': 'VERIFIED', 'occurrence_file_present': True, 'occurrence_bytes_verified_by_this_function': False}


## Verificar las 24 palabras y preparar una muestra

Esta comprobación exige igualdad exacta entre las medias calculadas y los prototipos guardados. Si falta el Parquet, informa del pendiente sin iniciar BERT.


In [4]:
VERIFICAR_MUESTRA = True
GUARDAR_MUESTRA_PARA_REVISION = True
if PARQUET.is_file() and VERIFICAR_MUESTRA:
    from datetime import datetime, timezone
    salida = None
    if GUARDAR_MUESTRA_PARA_REVISION:
        fecha = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
        salida = BASE / 'SalidasRevision' / f'MuestraContextual_{fecha}.zip'
    print(ctx.verificar_muestra(BASE, output=salida))
    if salida is not None: print('Archivo para adjuntar:', salida)
else:
    print('Muestra pendiente. Copia el Parquet a:', PARQUET)


{'status': 'VERIFIED_SAMPLE_ONLY', 'n_words': 24, 'n_occurrences': 2400, 'prototype_means_exact': True, 'full_occurrence_dataset_verified': False}
Archivo para adjuntar: C:\Users\jisaa\Documents\ProyectoTesis\EstudioContextual\GeneracionDataset\SalidasRevision\MuestraContextual_20261126T014231Z.zip


## Reproducir todos los prototipos desde las ocurrencias (opcional)

Lee las 10.302.900 ocurrencias guardadas y comprueba las dos huellas definitivas. No ejecuta BERT. Puede tardar por la lectura de 38,45 GiB; queda desactivado por defecto.


In [5]:
RECONSTRUIR_PROTOTIPOS = False
if RECONSTRUIR_PROTOTIPOS:
    destino = BASE / 'SalidasRegeneracion' / (ctx.STEM + '.npz')
    RECONSTRUIDOS = ctx.reconstruir_prototipos(BASE, output=destino)
    print('Reproducción exacta verificada:', destino)


## Regeneración desde corpus

El notebook separado GenerarOcurrencias.ipynb conserva la selección por hash, BERT y sus validaciones. Es una operación costosa y explícita. Para reutilizar los resultados existentes no es necesario ejecutarlo.
